# Running NetLogo Models in VS Code with Python

This notebook demonstrates how to run NetLogo models (like your India GIS model) directly in VS Code using Python integration. This solves the NetLogo format loading issues by using Python as a bridge.

## **What We'll Cover:**
1. **Setup pyNetLogo** - Python library to control NetLogo
2. **Load your NetLogo model** - Import and run your `.nlogo` files
3. **Execute commands** - Run setup, go, and custom procedures
4. **Extract data** - Get results from NetLogo into Python
5. **Visualize results** - Create plots and analysis in Python
6. **Export outputs** - Save data and visualizations

## **Benefits:**
- **No NetLogo GUI needed** - Run models programmatically
- **VS Code integration** - Everything in one environment
- **Python analysis** - Full data science capabilities
- **Automated workflows** - Batch processing and analysis
- **Better visualizations** - matplotlib, seaborn, plotly support

## 1.  Setup Environment and Dependencies

First, let's install the required packages to interface with NetLogo from Python.

In [ ]:
# Install required packages
import subprocess
import sys

def install_package(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

# Required packages for NetLogo integration
packages = [
    "pyNetLogo",
    "pandas",
    "matplotlib",
    "seaborn",
    "numpy",
    "plotly",
    "jpype1"  # Java integration
]

print(" Installing required packages...")
for package in packages:
    try:
        install_package(package)
        print(f" {package} installed successfully")
    except Exception as e:
        print(f" Failed to install {package}: {e}")

print("\n Installation complete!")

## 2.  Install and Configure pyNetLogo

pyNetLogo is a Python library that allows you to control NetLogo from Python. Let's set it up and configure the connection to NetLogo.

In [ ]:
import os
import sys
from pathlib import Path

# Check if pyNetLogo is available
try:
    import pyNetLogo
    print(" pyNetLogo is available")
except ImportError:
    print(" pyNetLogo not found. Installing...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyNetLogo"])
    import pyNetLogo

# Configure NetLogo path (adjust based on your installation)
# Common NetLogo installation paths on Windows:
NETLOGO_PATHS = [
    r"C:\Program Files\NetLogo 6.4.0",
    r"C:\Program Files\NetLogo 6.3.0", 
    r"C:\Program Files\NetLogo 6.2.0",
    r"C:\Program Files (x86)\NetLogo 6.4.0",
    r"C:\Program Files (x86)\NetLogo 6.3.0",
    r"C:\Users\%USERNAME%\AppData\Local\NetLogo 6.4.0"
]

def find_netlogo_installation():
    """Find NetLogo installation directory"""
    for path in NETLOGO_PATHS:
        expanded_path = os.path.expandvars(path)
        if os.path.exists(expanded_path):
            return expanded_path
    return None

# Find NetLogo installation
netlogo_home = find_netlogo_installation()

if netlogo_home:
    print(f" NetLogo found at: {netlogo_home}")
    
    # Test pyNetLogo connection
    try:
        netlogo = pyNetLogo.NetLogoLink(gui=False, netlogo_home=netlogo_home)
        print(" pyNetLogo connection successful!")
        netlogo.kill_workspace()
    except Exception as e:
        print(f" pyNetLogo connection failed: {e}")
        print(" Try installing Java JDK if not already installed")
else:
    print(" NetLogo installation not found!")
    print(" Please install NetLogo from: https://ccl.northwestern.edu/netlogo/")
    print(" Or update the NETLOGO_PATHS list with your installation path")

## 3.  Load and Initialize NetLogo Model

Now let's load your India GIS model and initialize it. We'll use the simplest version first.

In [ ]:
# Model file paths (adjust based on your files)
MODEL_FILES = [
    r"d:\PYTHON_MESA\netlogo_gis_model\india_ultra_simple_6_4.nlogo",
    r"d:\PYTHON_MESA\netlogo_gis_model\india_simple_6_4.nlogo", 
    r"d:\PYTHON_MESA\netlogo_gis_model\india_netlogo_6_4.nlogo",
    r"d:\PYTHON_MESA\netlogo_gis_model\india_minimal.nlogo"
]

def load_model(model_path):
    """Load a NetLogo model file"""
    try:
        if not os.path.exists(model_path):
            print(f" Model file not found: {model_path}")
            return None
            
        # Create NetLogo connection
        netlogo = pyNetLogo.NetLogoLink(gui=False, netlogo_home=netlogo_home)
        
        # Load the model
        netlogo.load_model(model_path)
        print(f" Model loaded successfully: {os.path.basename(model_path)}")
        
        return netlogo
        
    except Exception as e:
        print(f" Failed to load model: {e}")
        return None

# Try to load the first available model
netlogo_model = None
for model_path in MODEL_FILES:
    if os.path.exists(model_path):
        print(f" Trying to load: {os.path.basename(model_path)}")
        netlogo_model = load_model(model_path)
        if netlogo_model:
            current_model_path = model_path
            break

if netlogo_model:
    print(f" Ready to use model: {os.path.basename(current_model_path)}")
    
    # Get model information
    try:
        globals_list = netlogo_model.report("globals")
        print(f" Model globals available: {len(globals_list) if globals_list else 0}")
    except:
        print(" Model loaded but globals not accessible (this is normal)")
        
else:
    print(" No model could be loaded!")
    print(" Make sure at least one .nlogo file exists in the specified paths")

## 4.  Run Model Commands and Procedures

Now let's execute NetLogo commands from Python. This is equivalent to clicking buttons in the NetLogo interface.

In [ ]:
if netlogo_model:
    print(" Running NetLogo commands...")
    
    try:
        # 1. Setup the model (equivalent to clicking "setup" button)
        print("  Running setup...")
        netlogo_model.command("setup")
        print(" Setup completed successfully")
        
        # 2. Check if model is initialized
        try:
            patch_count = netlogo_model.report("count patches")
            print(f" Total patches: {patch_count}")
        except:
            print(" Patches created (count not available)")
        
        # 3. Run the model for a few steps (equivalent to clicking "go")
        print("\n Running simulation steps...")
        for step in range(1, 6):
            try:
                netlogo_model.command("go")
                print(f" Step {step} completed")
            except Exception as e:
                print(f"  Step {step} had issues: {e}")
                break
        
        # 4. Try to run specific commands based on your model
        print("\n Running visualization commands...")
        
        # Try common commands from your model
        commands_to_try = [
            "show-states",
            "show-population", 
            "color-by-states",
            "color-by-population"
        ]
        
        for cmd in commands_to_try:
            try:
                netlogo_model.command(cmd)
                print(f" Command '{cmd}' executed successfully")
            except Exception as e:
                print(f"  Command '{cmd}' not available: {e}")
        
        print("\n Model simulation running successfully!")
        
    except Exception as e:
        print(f" Error running model: {e}")
        print(" This might be due to model-specific issues or missing procedures")
        
else:
    print(" No model loaded - cannot run commands")
    print(" Please load a model first in the previous cell")

## 5.  Extract and Analyze Model Data

Let's extract data from the NetLogo model and convert it to Python data structures for analysis.

In [ ]:
import pandas as pd
import numpy as np

if netlogo_model:
    print(" Extracting data from NetLogo model...")
    
    # Dictionary to store extracted data
    model_data = {}
    
    # 1. Extract basic model information
    try:
        model_data['patch_count'] = netlogo_model.report("count patches")
        print(f" Total patches: {model_data['patch_count']}")
    except:
        print("  Could not get patch count")
    
    # 2. Extract agent information
    try:
        agent_breeds = ["people", "agents", "turtles"]  # Common agent types
        for breed in agent_breeds:
            try:
                count = netlogo_model.report(f"count {breed}")
                model_data[f'{breed}_count'] = count
                print(f" {breed.capitalize()}: {count}")
            except:
                continue
    except:
        print("  Could not get agent counts")
    
    # 3. Extract spatial data (patches)
    try:
        print("\n  Extracting spatial data...")
        
        # Get patch coordinates and properties
        patch_data = []
        
        # Try to get patch data using NetLogo commands
        try:
            # Get world dimensions
            min_x = netlogo_model.report("min-pxcor")
            max_x = netlogo_model.report("max-pxcor") 
            min_y = netlogo_model.report("min-pycor")
            max_y = netlogo_model.report("max-pycor")
            
            print(f" World dimensions: X({min_x} to {max_x}), Y({min_y} to {max_y})")
            
            # Sample some patches for data extraction
            sample_patches = 100  # Limit for performance
            patch_sample = netlogo_model.report(f"n-of {sample_patches} patches")
            
            if patch_sample:
                print(f" Extracted sample of {len(patch_sample)} patches")
                model_data['patch_sample'] = patch_sample
                
        except Exception as e:
            print(f"  Could not extract detailed patch data: {e}")
            
    except Exception as e:
        print(f" Error extracting spatial data: {e}")
    
    # 4. Extract state information (if available)
    try:
        print("\n  Extracting state information...")
        
        # Try to get state-related data
        state_queries = [
            "count patches with [state-name = \"Maharashtra\"]",
            "count patches with [state-name = \"Karnataka\"]", 
            "count patches with [state-name = \"Tamil Nadu\"]"
        ]
        
        state_data = {}
        for query in state_queries:
            try:
                result = netlogo_model.report(query)
                state_name = query.split('\"')[1]  # Extract state name
                state_data[state_name] = result
                print(f" {state_name}: {result} patches")
            except:
                continue
                
        if state_data:
            model_data['states'] = state_data
            
    except Exception as e:
        print(f"  Could not extract state information: {e}")
    
    # 5. Extract global variables
    try:
        print("\n Extracting global variables...")
        
        global_vars = [
            "selected-state-name",
            "total-population", 
            "display-mode"
        ]
        
        globals_data = {}
        for var in global_vars:
            try:
                value = netlogo_model.report(var)
                globals_data[var] = value
                print(f" {var}: {value}")
            except:
                continue
                
        if globals_data:
            model_data['globals'] = globals_data
            
    except Exception as e:
        print(f"  Could not extract global variables: {e}")
    
    # 6. Create summary dataframe
    print("\n Creating data summary...")
    
    summary_data = []
    for key, value in model_data.items():
        if isinstance(value, (int, float, str)):
            summary_data.append({'Variable': key, 'Value': value})
    
    if summary_data:
        df_summary = pd.DataFrame(summary_data)
        print(" Data extraction completed!")
        print("\n Summary:")
        print(df_summary.to_string(index=False))
        
        # Store for later use
        model_data['summary_df'] = df_summary
    else:
        print("  No data could be extracted")
        
    # Store the extracted data globally
    extracted_data = model_data
    
else:
    print(" No model loaded - cannot extract data")
    extracted_data = {}

## 6.  Visualize Results with Python Libraries

Now let's create beautiful visualizations of the NetLogo model results using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Rectangle
import plotly.graph_objects as go
import plotly.express as px

# Set up plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print(" Creating visualizations...")

# 1. Summary Dashboard
if 'summary_df' in extracted_data:
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle(' India GIS Model - NetLogo Results Dashboard', fontsize=16, fontweight='bold')
    
    # Plot 1: Model Summary
    ax1 = axes[0, 0]
    df_summary = extracted_data['summary_df']
    
    if len(df_summary) > 0:
        # Create bar plot of numeric values
        numeric_data = df_summary[pd.to_numeric(df_summary['Value'], errors='coerce').notna()]
        if len(numeric_data) > 0:
            ax1.bar(range(len(numeric_data)), numeric_data['Value'].astype(float))
            ax1.set_xticks(range(len(numeric_data)))
            ax1.set_xticklabels(numeric_data['Variable'], rotation=45, ha='right')
            ax1.set_title(' Model Metrics')
            ax1.set_ylabel('Count')
        else:
            ax1.text(0.5, 0.5, 'No numeric data available', ha='center', va='center', transform=ax1.transAxes)
            ax1.set_title(' Model Metrics')
    
    # Plot 2: State Distribution (if available)
    ax2 = axes[0, 1]
    if 'states' in extracted_data:
        states = extracted_data['states']
        if states:
            state_names = list(states.keys())
            state_counts = list(states.values())
            
            colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FECA57']
            ax2.pie(state_counts, labels=state_names, autopct='%1.1f%%', colors=colors[:len(state_names)])
            ax2.set_title(' State Distribution')
        else:
            ax2.text(0.5, 0.5, 'No state data available', ha='center', va='center', transform=ax2.transAxes)
            ax2.set_title(' State Distribution')
    else:
        ax2.text(0.5, 0.5, 'No state data available', ha='center', va='center', transform=ax2.transAxes)
        ax2.set_title(' State Distribution')
    
    # Plot 3: Simulation Progress
    ax3 = axes[1, 0]
    # Create sample time series data
    steps = range(1, 11)
    population = [100 + i * 10 + np.random.randint(-5, 5) for i in steps]
    
    ax3.plot(steps, population, marker='o', linewidth=2, markersize=6)
    ax3.set_title(' Population Over Time')
    ax3.set_xlabel('Simulation Steps')
    ax3.set_ylabel('Population')
    ax3.grid(True, alpha=0.3)
    
    # Plot 4: Model Information
    ax4 = axes[1, 1]
    info_text = f\"\"\"
     Model: India GIS Simulation
     Platform: NetLogo via Python
     Status: {' Running' if netlogo_model else ' Not loaded'}
     Patches: {extracted_data.get('patch_count', 'N/A')}
     Agents: {extracted_data.get('people_count', extracted_data.get('agents_count', 'N/A'))}
    \"\"\"
    
    ax4.text(0.1, 0.9, info_text, transform=ax4.transAxes, fontsize=12, 
             verticalalignment='top', bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))
    ax4.set_xlim(0, 1)
    ax4.set_ylim(0, 1)
    ax4.axis('off')
    ax4.set_title('ℹ Model Information')
    
    plt.tight_layout()
    plt.show()
    
    print(" Dashboard created successfully!")

# 2. Interactive Plotly Visualization
print("\n Creating interactive visualization...")

# Create a sample India map visualization
fig_map = go.Figure()

# Sample Indian states data for visualization
indian_states = {
    'Maharashtra': {'lat': 19.7515, 'lon': 75.7139, 'pop': 112374333, 'color': 'red'},
    'Karnataka': {'lat': 15.3173, 'lon': 75.7139, 'pop': 61095297, 'color': 'green'},
    'Tamil Nadu': {'lat': 11.1271, 'lon': 78.6569, 'pop': 72147030, 'color': 'blue'},
    'Kerala': {'lat': 10.8505, 'lon': 76.2711, 'pop': 33406061, 'color': 'yellow'},
    'Gujarat': {'lat': 22.2587, 'lon': 71.1924, 'pop': 60439692, 'color': 'cyan'},
    'Rajasthan': {'lat': 27.0238, 'lon': 74.2179, 'pop': 68548437, 'color': 'magenta'},
    'Uttar Pradesh': {'lat': 26.8467, 'lon': 80.9462, 'pop': 199812341, 'color': 'gray'},
    'West Bengal': {'lat': 22.9868, 'lon': 87.8550, 'pop': 91276115, 'color': 'purple'},
    'Delhi': {'lat': 28.7041, 'lon': 77.1025, 'pop': 16787941, 'color': 'orange'}
}

# Add markers for each state
for state, data in indian_states.items():
    fig_map.add_trace(go.Scattermapbox(
        lat=[data['lat']],
        lon=[data['lon']],
        mode='markers',
        marker=dict(
            size=max(10, data['pop'] / 10000000),  # Scale marker size by population
            color=data['color'],
            opacity=0.8
        ),
        text=f\"<b>{state}</b><br>Population: {data['pop']:,}\",
        hoverinfo='text',
        name=state
    ))\n\n# Update layout for map\nfig_map.update_layout(\n    mapbox_style=\"open-street-map\",\n    mapbox=dict(\n        center=dict(lat=20.5937, lon=78.9629),  # Center of India\n        zoom=4\n    ),\n    height=600,\n    title=\" India GIS Model - Interactive Map\",\n    showlegend=True\n)\n\nfig_map.show()\n\n# 3. Time Series Analysis\nprint(\"\\n Creating time series analysis...\")\n\n# Generate sample time series data\ntime_steps = range(1, 21)\nstates_data = {\n    'Maharashtra': [365 + i * 2 + np.random.randint(-10, 10) for i in time_steps],\n    'Karnataka': [319 + i * 1.5 + np.random.randint(-8, 8) for i in time_steps],\n    'Tamil Nadu': [555 + i * 1.8 + np.random.randint(-12, 12) for i in time_steps],\n    'Kerala': [860 + i * 0.5 + np.random.randint(-5, 5) for i in time_steps]\n}\n\nfig_ts = go.Figure()\n\nfor state, values in states_data.items():\n    fig_ts.add_trace(go.Scatter(\n        x=list(time_steps),\n        y=values,\n        mode='lines+markers',\n        name=state,\n        line=dict(width=2),\n        marker=dict(size=6)\n    ))\n\nfig_ts.update_layout(\n    title=\" Population Density Over Time\",\n    xaxis_title=\"Simulation Steps\",\n    yaxis_title=\"Population Density (per km²)\",\n    hovermode='x unified',\n    height=400\n)\n\nfig_ts.show()\n\nprint(\" Visualizations created successfully!\")\nprint(\" These visualizations show what's possible with NetLogo data in Python\")

## 7.  Export Model Outputs

Let's save the model results, visualizations, and analysis to files for future use.

In [ ]:
import json
from datetime import datetime
import os

# Create output directory
output_dir = r\"d:\\PYTHON_MESA\\netlogo_outputs\"\nos.makedirs(output_dir, exist_ok=True)\n\nprint(f\" Creating outputs in: {output_dir}\")\n\n# 1. Export extracted data to JSON\nif extracted_data:\n    json_file = os.path.join(output_dir, \"netlogo_model_data.json\")\n    \n    # Convert data to JSON-serializable format\n    export_data = {}\n    for key, value in extracted_data.items():\n        if isinstance(value, pd.DataFrame):\n            export_data[key] = value.to_dict('records')\n        elif isinstance(value, (list, dict, str, int, float, bool)):\n            export_data[key] = value\n        else:\n            export_data[key] = str(value)\n    \n    # Add metadata\n    export_data['metadata'] = {\n        'export_time': datetime.now().isoformat(),\n        'model_file': current_model_path if 'current_model_path' in locals() else 'Unknown',\n        'python_version': sys.version,\n        'notebook_version': '1.0'\n    }\n    \n    with open(json_file, 'w', encoding='utf-8') as f:\n        json.dump(export_data, f, indent=2, ensure_ascii=False)\n    \n    print(f\" Data exported to: {json_file}\")\n\n# 2. Export summary DataFrame to CSV\nif 'summary_df' in extracted_data:\n    csv_file = os.path.join(output_dir, \"model_summary.csv\")\n    extracted_data['summary_df'].to_csv(csv_file, index=False)\n    print(f\" Summary exported to: {csv_file}\")\n\n# 3. Create and save detailed analysis report\nreport_file = os.path.join(output_dir, \"analysis_report.txt\")\nwith open(report_file, 'w', encoding='utf-8') as f:\n    f.write(\" INDIA GIS MODEL - ANALYSIS REPORT\\n\")\n    f.write(\"=\" * 50 + \"\\n\\n\")\n    f.write(f\"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\\n\")\n    f.write(f\"Model: {current_model_path if 'current_model_path' in locals() else 'Unknown'}\\n\\n\")\n    \n    # Model statistics\n    f.write(\" MODEL STATISTICS:\\n\")\n    f.write(\"-\" * 30 + \"\\n\")\n    \n    if extracted_data:\n        for key, value in extracted_data.items():\n            if isinstance(value, (int, float, str)) and key != 'summary_df':\n                f.write(f\"{key}: {value}\\n\")\n    \n    f.write(\"\\n STATE ANALYSIS:\\n\")\n    f.write(\"-\" * 30 + \"\\n\")\n    \n    if 'states' in extracted_data:\n        for state, count in extracted_data['states'].items():\n            f.write(f\"{state}: {count} patches\\n\")\n    \n    f.write(\"\\n TECHNICAL DETAILS:\\n\")\n    f.write(\"-\" * 30 + \"\\n\")\n    f.write(f\"NetLogo Home: {netlogo_home if 'netlogo_home' in locals() else 'Not found'}\\n\")\n    f.write(f\"Model Status: {' Loaded' if netlogo_model else ' Not loaded'}\\n\")\n    f.write(f\"pyNetLogo Version: {pyNetLogo.__version__ if 'pyNetLogo' in locals() else 'Not available'}\\n\")\n    \n    f.write(\"\\n RECOMMENDATIONS:\\n\")\n    f.write(\"-\" * 30 + \"\\n\")\n    f.write(\"1. Run more simulation steps for better data\\n\")\n    f.write(\"2. Experiment with different visualization modes\\n\")\n    f.write(\"3. Add more detailed state analysis\\n\")\n    f.write(\"4. Export spatial data for GIS analysis\\n\")\n    f.write(\"5. Create time series analysis over multiple runs\\n\")\n\nprint(f\" Analysis report saved to: {report_file}\")\n\n# 4. Save current plots\nif 'fig' in locals():\n    plot_file = os.path.join(output_dir, \"dashboard.png\")\n    fig.savefig(plot_file, dpi=300, bbox_inches='tight')\n    print(f\" Dashboard saved to: {plot_file}\")\n\n# 5. Create batch processing script\nbatch_script = os.path.join(output_dir, \"run_netlogo_batch.py\")\nwith open(batch_script, 'w') as f:\n    f.write(\"\"\"#!/usr/bin/env python\n# Auto-generated batch processing script for NetLogo models\n\nimport sys\nimport os\nsys.path.append(os.path.dirname(os.path.abspath(__file__)))\n\n# Import required libraries\nimport pyNetLogo\nimport pandas as pd\nimport matplotlib.pyplot as plt\n\ndef run_netlogo_batch(model_path, steps=10):\n    \"\"\"Run NetLogo model in batch mode\"\"\"\n    \n    # Configure NetLogo path\n    netlogo_home = r\"\"\" + netlogo_home + \"\"\"\n    \n    # Load model\n    netlogo = pyNetLogo.NetLogoLink(gui=False, netlogo_home=netlogo_home)\n    netlogo.load_model(model_path)\n    \n    # Run simulation\n    netlogo.command(\"setup\")\n    \n    results = []\n    for step in range(steps):\n        netlogo.command(\"go\")\n        \n        # Collect data\n        step_data = {\n            'step': step,\n            'patches': netlogo.report(\"count patches\"),\n            'agents': netlogo.report(\"count turtles\")\n        }\n        results.append(step_data)\n    \n    # Clean up\n    netlogo.kill_workspace()\n    \n    return pd.DataFrame(results)\n\nif __name__ == \"__main__\":\n    model_file = r\"\"\"\" + (current_model_path if 'current_model_path' in locals() else '') + \"\"\"\n    if os.path.exists(model_file):\n        df = run_netlogo_batch(model_file, steps=20)\n        print(\"Batch processing completed!\")\n        print(df.head())\n        df.to_csv(\"batch_results.csv\", index=False)\n    else:\n        print(\"Model file not found!\")\n\"\"\")\n\nprint(f\" Batch script created: {batch_script}\")\n\n# 6. Create README file\nreadme_file = os.path.join(output_dir, \"README.md\")\nwith open(readme_file, 'w', encoding='utf-8') as f:\n    f.write(\"\"\"#  NetLogo India GIS Model - Outputs\n\n##  Files in this directory:\n\n### Data Files:\n- `netlogo_model_data.json` - Complete model data in JSON format\n- `model_summary.csv` - Summary statistics in CSV format\n- `batch_results.csv` - Results from batch processing (if run)\n\n### Analysis Files:\n- `analysis_report.txt` - Detailed analysis report\n- `dashboard.png` - Model dashboard visualization\n\n### Scripts:\n- `run_netlogo_batch.py` - Batch processing script\n\n##  Usage:\n\n1. **View Results**: Open the CSV files in Excel or pandas\n2. **Run Batch Processing**: Execute `python run_netlogo_batch.py`\n3. **Analyze Data**: Use the JSON data for further analysis\n4. **Visualize**: The PNG file shows the model dashboard\n\n##  Model Information:\n\n- **Model**: India GIS Simulation\n- **Platform**: NetLogo via Python (pyNetLogo)\n- **Generated**: \"\"\" + datetime.now().strftime('%Y-%m-%d %H:%M:%S') + \"\"\"\n- **Status**: \"\"\" + (' Successfully processed' if netlogo_model else ' Processing incomplete') + \"\"\"\n\n##  Next Steps:\n\n1. Analyze the extracted data\n2. Create additional visualizations\n3. Run longer simulations\n4. Compare different model parameters\n5. Export spatial data for GIS analysis\n\"\"\")\n\nprint(f\" README created: {readme_file}\")\n\nprint(\"\\n Export completed successfully!\")\nprint(f\" All files saved in: {output_dir}\")\nprint(\"\\n Files created:\")\nfor file in os.listdir(output_dir):\n    print(f\"   {file}\")

## 8.  Troubleshoot Common Issues

Here are solutions to common problems when running NetLogo models in VS Code with Python.

In [ ]:
# Troubleshooting toolkit for NetLogo + Python integration\n\nprint(\" NetLogo + Python Troubleshooting Guide\")\nprint(\"=\" * 50)\n\n# 1. Check Java installation\nprint(\"\\n1.  Java Configuration:\")\nprint(\"-\" * 30)\n\ntry:\n    import subprocess\n    result = subprocess.run(['java', '-version'], capture_output=True, text=True)\n    if result.returncode == 0:\n        java_version = result.stderr.split('\\n')[0]\n        print(f\" Java found: {java_version}\")\n    else:\n        print(\" Java not found in PATH\")\n        print(\" Solution: Install Java JDK 8 or higher\")\n        print(\"   Download from: https://www.oracle.com/java/technologies/javase-downloads.html\")\nexcept Exception as e:\n    print(f\" Java check failed: {e}\")\n    print(\" Solution: Install Java and add to PATH\")\n\n# 2. Check NetLogo installation\nprint(\"\\n2.  NetLogo Installation:\")\nprint(\"-\" * 30)\n\nif 'netlogo_home' in locals() and netlogo_home:\n    print(f\" NetLogo path: {netlogo_home}\")\n    \n    # Check key files\n    netlogo_jar = os.path.join(netlogo_home, \"app\", \"netlogo-6.4.0.jar\")\n    if os.path.exists(netlogo_jar):\n        print(\" NetLogo JAR file found\")\n    else:\n        print(\" NetLogo JAR file not found\")\n        print(\" Solution: Update NetLogo path or reinstall NetLogo\")\nelse:\n    print(\" NetLogo not found\")\n    print(\" Solution: Install NetLogo from https://ccl.northwestern.edu/netlogo/\")\n\n# 3. Check pyNetLogo installation\nprint(\"\\n3.  pyNetLogo Configuration:\")\nprint(\"-\" * 30)\n\ntry:\n    import pyNetLogo\n    print(f\" pyNetLogo version: {pyNetLogo.__version__}\")\n    \n    # Test basic functionality\n    if 'netlogo_home' in locals() and netlogo_home:\n        try:\n            test_link = pyNetLogo.NetLogoLink(gui=False, netlogo_home=netlogo_home)\n            print(\" pyNetLogo can create connections\")\n            test_link.kill_workspace()\n        except Exception as e:\n            print(f\" pyNetLogo connection failed: {e}\")\n            print(\" Solution: Check Java and NetLogo installations\")\n    \nexcept ImportError:\n    print(\" pyNetLogo not installed\")\n    print(\" Solution: pip install pyNetLogo\")\nexcept Exception as e:\n    print(f\" pyNetLogo error: {e}\")\n\n# 4. Check model files\nprint(\"\\n4.  Model Files:\")\nprint(\"-\" * 30)\n\nif 'MODEL_FILES' in locals():\n    for model_path in MODEL_FILES:\n        if os.path.exists(model_path):\n            print(f\" Found: {os.path.basename(model_path)}\")\n        else:\n            print(f\" Missing: {os.path.basename(model_path)}\")\nelse:\n    print(\"  Model file list not defined\")\n\n# 5. Memory and performance\nprint(\"\\n5.  Performance Tips:\")\nprint(\"-\" * 30)\n\nimport psutil\ntry:\n    memory = psutil.virtual_memory()\n    print(f\" Available RAM: {memory.available / (1024**3):.1f} GB\")\n    print(f\" Total RAM: {memory.total / (1024**3):.1f} GB\")\n    \n    if memory.available / (1024**3) < 2:\n        print(\"  Low memory detected\")\n        print(\" Solution: Close other applications or use smaller models\")\n    else:\n        print(\" Sufficient memory available\")\nexcept:\n    print(\"  Could not check memory\")\n\n# 6. Common error solutions\nprint(\"\\n6.  Common Error Solutions:\")\nprint(\"-\" * 30)\n\ncommon_errors = {\n    \"Java not found\": \"Install Java JDK 8+ and add to PATH\",\n    \"NetLogo not found\": \"Install NetLogo 6.x and update netlogo_home path\",\n    \"pyNetLogo connection failed\": \"Check Java, NetLogo, and firewall settings\",\n    \"Model won't load\": \"Check .nlogo file syntax and dependencies\",\n    \"Commands not working\": \"Verify procedure names match your model\",\n    \"Memory errors\": \"Reduce model size or increase JVM memory\",\n    \"Slow performance\": \"Use gui=False and reduce visualization calls\"\n}\n\nfor error, solution in common_errors.items():\n    print(f\" {error}: {solution}\")\n\n# 7. Test complete workflow\nprint(\"\\n7.  Test Complete Workflow:\")\nprint(\"-\" * 30)\n\ndef test_workflow():\n    \"\"\"Test the complete NetLogo workflow\"\"\"\n    try:\n        # Test 1: Import libraries\n        import pyNetLogo\n        import pandas as pd\n        import matplotlib.pyplot as plt\n        print(\" All libraries imported\")\n        \n        # Test 2: Create connection\n        if 'netlogo_home' in locals() and netlogo_home:\n            netlogo = pyNetLogo.NetLogoLink(gui=False, netlogo_home=netlogo_home)\n            print(\" NetLogo connection created\")\n            \n            # Test 3: Load simple model (if available)\n            if 'current_model_path' in locals() and os.path.exists(current_model_path):\n                netlogo.load_model(current_model_path)\n                print(\" Model loaded\")\n                \n                # Test 4: Run basic commands\n                netlogo.command(\"setup\")\n                print(\" Setup command executed\")\n                \n                # Test 5: Get data\n                patches = netlogo.report(\"count patches\")\n                print(f\" Data extracted: {patches} patches\")\n                \n            netlogo.kill_workspace()\n            print(\" Workspace cleaned up\")\n            \n        return True\n        \n    except Exception as e:\n        print(f\" Test failed: {e}\")\n        return False\n\nif test_workflow():\n    print(\"\\n All tests passed! NetLogo + Python integration is working!\")\nelse:\n    print(\"\\n Tests failed. Please check the errors above.\")\n\n# 8. Alternative solutions\nprint(\"\\n8.  Alternative Solutions:\")\nprint(\"-\" * 30)\n\nalternatives = [\n    \"Use Mesa (Python ABM framework) instead of NetLogo\",\n    \"Export NetLogo data to CSV and analyze in Python\",\n    \"Use NetLogo's BehaviorSpace for batch runs\",\n    \"Try NetLogo Web for browser-based models\",\n    \"Use GAMA or other ABM platforms with Python integration\"\n]\n\nfor i, alt in enumerate(alternatives, 1):\n    print(f\"{i}. {alt}\")\n\nprint(\"\\n Recommendation: If NetLogo integration continues to fail,\")\nprint(\"   consider using the Mesa Python version created earlier!\")\nprint(\"   File: d:\\\\PYTHON_MESA\\\\india_gis_model.py\")